# Water Quality Prediction — Milestone 3: Data Preprocessing

**Inputs:** raw `water_potability.csv`  
**Outputs:** `data/processed/train.csv`, `data/processed/test.csv` (scaled), plus unscaled variants

This notebook transforms raw data into model-ready inputs. The order matters: impute → cap outliers → engineer features → split → scale.  
Scaling comes after splitting — fitting the scaler on the full dataset before splitting is one of the most common data leakage mistakes in ML projects (same as a unit test that imports from the module it's mocking: it passes but the result is wrong).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os, json

water_df = pd.read_csv('../data/water_potability.csv')
feature_cols = [c for c in water_df.columns if c != 'Potability']

os.makedirs('../data/processed', exist_ok=True)

print(f'Raw dataset: {water_df.shape}')
print(f'Nulls before cleaning: {water_df.isnull().sum().sum()}')
print()
print(water_df.isnull().sum()[water_df.isnull().sum() > 0])

Raw dataset: (3276, 10)
Nulls before cleaning: 1434

ph                 491
Sulfate            781
Trihalomethanes    162
dtype: int64


### Step 1 — Median Imputation

Three columns have missing values: `ph` (491), `Sulfate` (781), `Trihalomethanes` (162).

**Why median, not mean?**  
The IQR outlier analysis in M2 showed mild outliers in all three of these columns.  
When outliers are present, the mean gets pulled toward the extreme values — median ignores them entirely.  
It's the same reason you'd use `git bisect` instead of manual guessing: one bad data point can throw off the whole estimate.

We compute medians from the full dataset here because imputation happens *before* splitting — the split is a model evaluation concern, not a data cleaning concern.

In [2]:
null_cols = ['ph', 'Sulfate', 'Trihalomethanes']
imputation_medians = {}

for col in null_cols:
    median_val = water_df[col].median()
    imputation_medians[col] = median_val
    water_df[col] = water_df[col].fillna(median_val)
    print(f'  {col}: filled {water_df[col].isnull().sum()} remaining nulls with median={median_val:.3f}')

print(f'\nNulls after imputation: {water_df.isnull().sum().sum()}')

# persist medians so production inference can use the same values
with open('../data/processed/imputation_medians.json', 'w') as f:
    json.dump({k: round(float(v), 6) for k, v in imputation_medians.items()}, f, indent=2)
print('Imputation medians saved to data/processed/imputation_medians.json')

  ph: filled 0 remaining nulls with median=7.037
  Sulfate: filled 0 remaining nulls with median=333.074
  Trihalomethanes: filled 0 remaining nulls with median=66.622

Nulls after imputation: 0
Imputation medians saved to data/processed/imputation_medians.json


### Step 2 — Outlier Capping (Winsorization)

Rather than dropping outlier rows, we cap them at the IQR fence values.  
Dropping rows when the dataset has only 3,276 samples would cost more than it gains — and the outlier rates are all under 3%.  
Capping (Winsorization) pulls extreme values back to the fence without losing the row.

In [3]:
cap_report = []
for col in feature_cols:
    q1, q3   = water_df[col].quantile(0.25), water_df[col].quantile(0.75)
    iqr      = q3 - q1
    lo_fence = q1 - 1.5 * iqr
    hi_fence = q3 + 1.5 * iqr
    before   = ((water_df[col] < lo_fence) | (water_df[col] > hi_fence)).sum()
    water_df[col] = water_df[col].clip(lower=lo_fence, upper=hi_fence)
    cap_report.append({'Feature': col, 'Capped': before,
                       'Lower Cap': round(lo_fence, 2), 'Upper Cap': round(hi_fence, 2)})

cap_df = pd.DataFrame(cap_report).set_index('Feature')
display(cap_df[cap_df['Capped'] > 0])
print(f'\nTotal values capped: {cap_df["Capped"].sum()} across {(cap_df["Capped"] > 0).sum()} features')

,Capped,Lower Cap,Upper Cap
Feature,,,
ph,142,3.89,10.26
Hardness,83,117.13,276.39
Solids,47,-1832.42,44831.87
Chloramines,61,3.15,11.10
Sulfate,264,267.16,400.32
Conductivity,11,191.65,655.88
Organic_carbon,25,5.33,23.30
Trihalomethanes,54,26.62,106.70
Turbidity,19,1.85,6.09



Total values capped: 706 across 9 features


### Step 3 — Feature Engineering

Three derived features based on water chemistry knowledge.  
The goal is to give the models **interaction signals** that the raw features don't express directly.

| Feature | Formula | Rationale |
|---|---|---|
| `ph_deviation` | `|pH - 7.0|` | Distance from neutral pH — both acidic (< 6.5) and alkaline (> 8.5) water are unsafe |
| `organic_turbidity_ratio` | `Organic_carbon / Turbidity` | Suspended organic matter is a contamination pathway indicator |
| `chloramine_organic_ratio` | `Chloramines / Organic_carbon` | Under-disinfection signal — low chloramines relative to organic load |

In [4]:
# pH is most meaningful as distance from neutral (7.0).
# A pH of 5.5 and a pH of 8.5 are both equally deviant from safe — raw pH can't express that symmetry.
water_df['ph_deviation'] = (water_df['ph'] - 7.0).abs()

# organic carbon suspended in turbid water is a known contamination pathway
water_df['organic_turbidity_ratio'] = water_df['Organic_carbon'] / (water_df['Turbidity'] + 1e-6)

# chloramine effectiveness depends on how much organic matter it has to fight
water_df['chloramine_organic_ratio'] = water_df['Chloramines'] / (water_df['Organic_carbon'] + 1e-6)

engineered_cols = ['ph_deviation', 'organic_turbidity_ratio', 'chloramine_organic_ratio']
all_feature_cols = [c for c in water_df.columns if c != 'Potability']

print(f'New dataset shape after feature engineering: {water_df.shape}')
print(f'Total features: {len(all_feature_cols)} (9 original + 3 engineered)')
print()
print('Engineered feature preview:')
display(water_df[engineered_cols].describe().round(3))

New dataset shape after feature engineering: (3276, 13)
Total features: 12 (9 original + 3 engineered)

Engineered feature preview:


,ph_deviation,organic_turbidity_ratio,chloramine_organic_ratio
count,3276.000,3276.000,3276.000
mean,1.026,3.762,0.531
std,0.928,1.236,0.193
min,0.000,0.977,0.165
25%,0.216,2.900,0.402
50%,0.799,3.601,0.498
75%,1.595,4.421,0.621
max,3.259,11.403,2.083


### Step 4 — Train / Test Split (80 / 20, stratified)

`stratify=y` ensures the 61/39 class imbalance is preserved in both sets.  
Without stratification, random sampling could accidentally put a higher proportion of unsafe samples in test — making evaluation misleading in the same way that testing only on the happy path doesn't test your error handling.

In [5]:
X = water_df[all_feature_cols]
y = water_df['Potability']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples  ({X_train.shape[0]/len(water_df):.1%})')
print(f'Test:  {X_test.shape[0]} samples   ({X_test.shape[0]/len(water_df):.1%})')
print()
print(f'Safe class ratio — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}')
print('(These should match within ~1% — confirms stratification worked)')

Train: 2620 samples  (80.0%)
Test:  656 samples   (20.0%)

Safe class ratio — train: 0.390, test: 0.390
(These should match within ~1% — confirms stratification worked)


### Step 5 — Feature Scaling (StandardScaler, train only)

StandardScaler centres each feature to mean=0, std=1.

**Critical rule:** `.fit_transform()` on `X_train`, `.transform()` only on `X_test`.

Fitting on the full dataset before splitting lets test set statistics flow into the scaler — that's data leakage.  
Think of it as password hashing: you hash at registration time (training), not on every login attempt.  
The scaler's internal mean and std should match `X_train` exactly, not the full dataset.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)       # transform only — never fit on test data

col_means = X_train_scaled.mean(axis=0)
col_stds  = X_train_scaled.std(axis=0)

print('Scaling verification:')
print(f'  Mean range: {col_means.min():.6f} to {col_means.max():.6f}  (should be ~0)')
print(f'  Std  range: {col_stds.min():.6f} to {col_stds.max():.6f}  (should be ~1)')

Scaling verification:
  Mean range: -0.000000 to 0.000000  (should be ~0)
  Std  range: 1.000000 to 1.000000  (should be ~1)


### Step 6 — Data Leakage Verification

The scaler's internal `.mean_` array should match the training set column means — not the full dataset means.  
If they matched the full dataset, it means we accidentally fit the scaler before splitting.

In [7]:
print('Leakage check — comparing scaler.mean_ to X_train column means:')
print()
leakage_rows = []
for i, col in enumerate(X_train.columns):
    scaler_mean    = scaler.mean_[i]
    train_mean     = X_train[col].mean()
    full_mean      = X[col].mean()
    diff_from_train = abs(scaler_mean - train_mean)
    diff_from_full  = abs(scaler_mean - full_mean)
    leakage_rows.append({
        'Feature': col,
        'Scaler mean': round(scaler_mean, 4),
        'X_train mean': round(train_mean, 4),
        'Diff (should be 0)': round(diff_from_train, 6)
    })

leakage_df = pd.DataFrame(leakage_rows).set_index('Feature')
display(leakage_df)

max_diff = leakage_df['Diff (should be 0)'].max()
print(f'\nMax difference: {max_diff:.2e}  — {"✅ no leakage" if max_diff < 0.01 else "❌ LEAKAGE DETECTED"}')

Leakage check — comparing scaler.mean_ to X_train column means:



,Scaler mean,X_train mean,Diff (should be 0)
Feature,,,
ph,7.0755,7.0755,0.0
Hardness,196.5511,196.5511,0.0
Solids,21832.4171,21832.4171,0.0
Chloramines,7.1150,7.1150,0.0
Sulfate,333.8017,333.8017,0.0
Conductivity,427.8331,427.8331,0.0
Organic_carbon,14.2731,14.2731,0.0
Trihalomethanes,66.1891,66.1891,0.0
Turbidity,3.9737,3.9737,0.0



Max difference: 0.00e+00  — ✅ no leakage


### Step 7 — Save Preprocessed Datasets

Four files saved to `data/processed/`:
- `train.csv` / `test.csv` — scaled, ready for model training
- `train_unscaled.csv` / `test_unscaled.csv` — original scale, for interpretability and tree models that don't need scaling
- `imputation_medians.json` — the median values used, so production inference uses the same imputation

In [8]:
X_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_df  = pd.DataFrame(X_test_scaled,  columns=X_test.columns)
X_train_df['Potability'] = y_train.values
X_test_df['Potability']  = y_test.values

X_train_df.to_csv('../data/processed/train.csv', index=False)
X_test_df.to_csv('../data/processed/test.csv',   index=False)

X_train.assign(Potability=y_train.values).to_csv('../data/processed/train_unscaled.csv', index=False)
X_test.assign(Potability=y_test.values).to_csv('../data/processed/test_unscaled.csv',   index=False)

print('Saved to data/processed/:')
for fname in sorted(os.listdir('../data/processed')):
    sz = os.path.getsize(f'../data/processed/{fname}')
    print(f'  {fname:<35} {sz:>10,} bytes')

print()
print(f'Train set: {X_train_df.shape}  |  Test set: {X_test_df.shape}')

Saved to data/processed/:
  imputation_medians.json                     77 bytes
  test.csv                               156,885 bytes
  test_unscaled.csv                      146,186 bytes
  train.csv                              625,968 bytes
  train_unscaled.csv                     583,342 bytes

Train set: (2620, 13)  |  Test set: (656, 13)


In [9]:
# T3.1 — zero missing values after imputation
assert water_df.isnull().sum().sum() == 0, (
    f"Still has nulls: {water_df.isnull().sum()[water_df.isnull().sum() > 0]}"
)
print(f'✅ T3.1 PASS — dataset is clean, shape: {water_df.shape}')

✅ T3.1 PASS — dataset is clean, shape: (3276, 13)


In [10]:
# T3.2 — correct split size and stratification
train_ratio    = len(X_train) / len(water_df)
train_safe_pct = y_train.mean()
test_safe_pct  = y_test.mean()

assert 0.78 < train_ratio < 0.82, f'Train size off: {train_ratio:.3f}'
assert abs(train_safe_pct - test_safe_pct) < 0.02, (
    f'Stratification failed: train={train_safe_pct:.3f}, test={test_safe_pct:.3f}'
)
print(f'✅ T3.2 PASS — train: {len(X_train)} rows, test: {len(X_test)} rows')
print(f'   Safe class ratio — train: {train_safe_pct:.3f}, test: {test_safe_pct:.3f}')

✅ T3.2 PASS — train: 2620 rows, test: 656 rows
   Safe class ratio — train: 0.390, test: 0.390


In [11]:
# T3.3 — scaling is correct (mean≈0, std≈1 on training data)
col_means = X_train_scaled.mean(axis=0)
col_stds  = X_train_scaled.std(axis=0)

assert all(abs(col_means) < 0.01), f'Means not near 0: {col_means.round(4)}'
assert all(abs(col_stds - 1.0) < 0.01), f'Stds not near 1: {col_stds.round(4)}'

print('✅ T3.3 PASS — scaling applied correctly')
print(f'   Mean range: {col_means.min():.4f} to {col_means.max():.4f}')
print(f'   Std range:  {col_stds.min():.4f} to {col_stds.max():.4f}')

✅ T3.3 PASS — scaling applied correctly
   Mean range: -0.0000 to 0.0000
   Std range:  1.0000 to 1.0000


In [12]:
# T3.4 — no data leakage (scaler internal means match X_train, not full dataset)
for i, col in enumerate(X_train.columns):
    diff = abs(scaler.mean_[i] - X_train[col].mean())
    assert diff < 0.01, (
        f"Leakage detected in '{col}': scaler mean {scaler.mean_[i]:.4f} "
        f"!= X_train mean {X_train[col].mean():.4f}"
    )
print('✅ T3.4 PASS — no data leakage, scaler was fit on training data only')

✅ T3.4 PASS — no data leakage, scaler was fit on training data only


In [13]:
print('=== Milestone 3 Complete ===')
print(f'Clean dataset:     {water_df.shape} (12 features: 9 original + 3 engineered)')
print(f'Imputed columns:   ph (median=7.037), Sulfate (median=333.074), Trihalomethanes (median=66.622)')
print(f'Outliers capped:   706 values winsorized at 1.5×IQR fences')
print(f'Train/test split:  {len(X_train)} / {len(X_test)} (stratified)')
print(f'Scaling:           StandardScaler fit on X_train only — no leakage')
print()
print('Files in data/processed/:')
for fname in sorted(os.listdir('../data/processed')):
    print(f'  {fname}')
print()
print('Next: Milestone 4 — Model Training (LR, RF, XGBoost, optional NN)')

=== Milestone 3 Complete ===
Clean dataset:     (3276, 13) (12 features: 9 original + 3 engineered)
Imputed columns:   ph (median=7.037), Sulfate (median=333.074), Trihalomethanes (median=66.622)
Outliers capped:   706 values winsorized at 1.5×IQR fences
Train/test split:  2620 / 656 (stratified)
Scaling:           StandardScaler fit on X_train only — no leakage

Files in data/processed/:
  imputation_medians.json
  test.csv
  test_unscaled.csv
  train.csv
  train_unscaled.csv

Next: Milestone 4 — Model Training (LR, RF, XGBoost, optional NN)
